# 01_prompt_llama
This prompt was desined to have a higher recall in llama, it was derived from previous "Prompt 13 (Safety): Avg ~3.2 (High Recall)".

In [9]:
PROMPT_TEXT = r'''
def build_prompt(
    job_desc: str,
    tasks_str: str,
    clean_titles: list[str],
    domain: str,
    job_ad_title: str,
    job_sector_category: str,
    full_ad_text: str,
) -> str:
    n_candidates = len(clean_titles)
    numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(clean_titles))
    
    full_excerpt = full_ad_text.strip()[:700] if full_ad_text else ""
    full_block = f"\nFULL AD EXCERPT (use for concrete tools/duties only):\n{full_excerpt}\n" if full_excerpt else ""

    return f"""
TASK
You are a Senior Labor Market Economist auditing SOC matches. Evaluate candidates and DROP those that are NOT functional matches for the job.

GOAL: THE UP TO 4 TARGET
There are {n_candidates} candidates.
- Your goal is to keep 2 to 4 candidates for every job.
- Assume that at least 2 candidates should normally be KEPT.
- Only keep 1 if you are really sure all others are CLEARLY wrong.
- If more than 4 are valid, drop the most generic ones to fit the 4-candidate cap.

DEFAULT BEHAVIOUR
- When in doubt, KEEP rather than DROP, as long as the candidate is in the correct functional family.
- Actively look for a SECOND and THIRD valid match before dropping.

JOB CONTEXT (SOURCE OF TRUTH)
Title: {job_ad_title}
Sector/Domain: {job_sector_category} | {domain}
Functional Tasks: {tasks_str}
{full_block}

CANDIDATES (1-based)
{numbered}

ANCHOR PROTECTION (FUNCTION FIRST)
1) The ANCHOR is the candidate whose FUNCTION best matches the title’s role type (not seniority, not niche specialism).
2) You MUST KEEP the ANCHOR unless CORE EVIDENCE explicitly contradicts it.
3) TITLE KEYWORD LOCK: If the job title contains a functional keyword (e.g. driver, nurse, electrician, developer, sommelier), and a candidate directly matching that function exists, you MUST keep it.

KEEPING LOGIC (JOB MIX RECALL)
After keeping the ANCHOR, you SHOULD normally keep 2–4 additional candidates.

Keep an additional candidate if ANY apply:
- Same functional family (e.g. sommelier <-> waiter; care worker <-> home health aide).
- Same occupation at a different level (e.g. engineer <-> technologist/technician).
- Tasks partially overlap or describe adjacent duties.
- Do NOT keep roles that are purely generic or cross-sector (e.g. general managers, administrators, laborers) unless the tasks clearly align.
- The role plausibly spans more than one SOC in real labour markets.

HIERARCHY & IT GATES
1) MANAGER RULE
   - If the title does NOT include Manager/Lead/Director, only keep a managerial candidate if tasks explicitly describe staff oversight, rotas, or budgeting.
   - Do NOT keep managerial SOCs based on seniority alone.

2) IT DOMAIN LOCK
   - If the title is explicitly IT, do NOT drop the IT anchor regardless of domain.
   - For non-IT domains, keep tech roles if:
     - tasks mention specific tools (Python, SQL, APIs, etc), OR
     - the title strongly implies a digital/technical function (analyst, systems, platform, data).

RE-ANCHOR (FINAL CHECK)
Title: {job_ad_title}
Tasks: {tasks_str}
Count: {n_candidates} candidates.
You should keep 2–4 candidates.
Keeping only 1 should be rare and requires clear mismatch for all others.

OUTPUT
Return ONLY a valid JSON object with exactly one key: "drop".
Schema:
{{"drop":[...]}}
""".strip()
'''.strip()


# run

bge e5 and gte

In [11]:
import subprocess, re
from pathlib import Path

SBATCH = Path("/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/sbatch/run_llama_drop_bge.sbatch")
assert SBATCH.exists(), f"Missing sbatch file: {SBATCH}"

out = subprocess.check_output(["sbatch", str(SBATCH)], text=True)
jobid = re.search(r"(\d+)", out).group(1)
print("JOBID:", jobid)
bge_jobid = jobid

import subprocess, re
from pathlib import Path

SBATCH = Path("/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/sbatch/run_llama_drop_e5.sbatch")
assert SBATCH.exists(), f"Missing sbatch file: {SBATCH}"

out = subprocess.check_output(["sbatch", str(SBATCH)], text=True)
jobid = re.search(r"(\d+)", out).group(1)
print("JOBID:", jobid)
e5_jobid = jobid

import subprocess, re
from pathlib import Path

SBATCH = Path("/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/sbatch/run_llama_drop_gte.sbatch")
assert SBATCH.exists(), f"Missing sbatch file: {SBATCH}"

out = subprocess.check_output(["sbatch", str(SBATCH)], text=True)
jobid = re.search(r"(\d+)", out).group(1)
print("JOBID:", jobid)
gte_jobid = jobid

JOBID: 2206978
JOBID: 2206979
JOBID: 2206980


# monitor run

In [13]:
import time, subprocess
from pathlib import Path

LOG_DIR = Path("/projects/a5u/adu_dev/aisi-economy-index/logs")
def sh(cmd):
    return subprocess.run(cmd, shell=True, text=True, capture_output=True).stdout.strip()
while True:
    print("\n=== squeue ===")
    sq = sh(f"squeue -j {jobid}")
    print(sq if sq else "<finished>")

    out_logs = sorted(LOG_DIR.glob(f"*{jobid}*.out"))
    err_logs = sorted(LOG_DIR.glob(f"*{jobid}*.err"))

    if out_logs:
        print("\n--- STDOUT ---")
        print(sh(f"tail -n 20 {out_logs[-1]}"))

    if err_logs:
        print("\n--- STDERR ---")
        print(sh(f"tail -n 20 {err_logs[-1]}"))

    if not sq:
        break

    time.sleep(5)



=== squeue ===
JOBID         USER PARTITION                     NAME ST TIME_LIMIT       TIME  TIME_LEFT NODES NODELIST(REASON)

--- STDOUT ---
NPZ_DIR=/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection/gte_large
total 7,1M
drwxr-xr-x 2 autonomyluiz.a5u autonomyluiz.a5u 4,0K fev  8 22:26 adzuna_month01
-rw-r--r-- 1 autonomyluiz.a5u autonomyluiz.a5u 569K fev  5 16:02 adzuna_month01.npz
drwxr-xr-x 2 autonomyluiz.a5u autonomyluiz.a5u 4,0K fev  8 22:26 adzuna_month02
-rw-r--r-- 1 autonomyluiz.a5u autonomyluiz.a5u 570K fev  5 16:03 adzuna_month02.npz
drwxr-xr-x 2 autonomyluiz.a5u autonomyluiz.a5u 4,0K fev  8 22:26 adzuna_month03
-rw-r--r-- 1 autonomyluiz.a5u autonomyluiz.a5u 571K fev  5 16:03 adzuna_month03.npz
drwxr-xr-x 2 autonomyluiz.a5u autonomyluiz.a5u 4,0K fev  8 22:26 adzuna_month04
-rw-r--r-- 1 autonomyluiz.a5u autonomyluiz.a5u 571K fev  5 16:04 adzuna_month04.npz
drwxr-xr-x 2 autonomyluiz.a5u autonomyluiz.a5u 4,0K fev  8 

# Report

In [14]:
import os
os.getcwd()
import importlib
import gen_report_helper as grh # gen_report_helper.py 
from pathlib import Path
importlib.reload(grh) 

res = grh.generate_LLM_MODEL_full_report_and_plots(
    JSONL_BASE_DIR=Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection/"),
    NPZ_BASE_DIR=Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection"),
    llm_model="meta-llama--Meta-Llama-3.1-8B-Instruct",
    prompt_text=PROMPT_TEXT,
    reports_subdir="reports/llama_reports"
)
res


[OK] Wrote:
  txt:     /lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/reports/llama_reports/20260208_223700/20260208_223700_FULL_TEXT_REPORT.txt
  heatmap: /lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/reports/llama_reports/20260208_223700/20260208_223700_avg_jaccard_heatmap.png
  venn:    /lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/reports/llama_reports/20260208_223700/20260208_223700_venn_kept_pairs_pooled.png


{'txt_path': '/lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/reports/llama_reports/20260208_223700/20260208_223700_FULL_TEXT_REPORT.txt',
 'heatmap_path': '/lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/reports/llama_reports/20260208_223700/20260208_223700_avg_jaccard_heatmap.png',
 'venn_path': '/lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/reports/llama_reports/20260208_223700/20260208_223700_venn_kept_pairs_pooled.png',
 'out_dir': '/lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/reports/llama_reports/20260208_223700',
 'timestamp': '20260208_223700',
 'loaded_months_for_plots': 14,
 'cwd': '/lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev'}

In [ ]:
#!/usr/bin/env python3
"""
Validate embedding file paths and check normalization status.
"""
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List

def check_norms(path: str, key: Optional[str] = None) -> Dict:
    """
    Check if embeddings exist and their normalization status.
    
    Args:
        path: Path to .npy or .npz file
        key: Key name if npz file (e.g., 'embeddings', 'role_embeds')
    
    Returns:
        Dict with validation results
    """
    p = Path(path)
    result = {
        "path": path,
        "exists": False,
        "normalized": None,
        "shape": None,
        "dtype": None,
        "mean_norm": None,
        "error": None
    }
    
    try:
        if not p.exists():
            result["error"] = "File not found"
            return result
        
        result["exists"] = True
        
        # Load embeddings
        if path.endswith(".npy"):
            emb = np.load(path)
        elif path.endswith(".npz"):
            if key is None:
                result["error"] = "NPZ file requires 'key' argument"
                return result
            data = np.load(path, allow_pickle=True)
            if key not in data.files:
                result["error"] = f"Key '{key}' not in NPZ. Available: {data.files}"
                return result
            emb = data[key]
        else:
            result["error"] = "Unknown file type (not .npy or .npz)"
            return result
        
        result["shape"] = emb.shape
        result["dtype"] = str(emb.dtype)
        
        # Check normalization (sample first 1000 rows for speed)
        sample_size = min(1000, len(emb))
        sample = emb[:sample_size].astype(np.float32)
        norms = np.linalg.norm(sample, axis=1)
        mean_norm = float(np.mean(norms))
        std_norm = float(np.std(norms))
        
        result["mean_norm"] = mean_norm
        result["std_norm"] = std_norm
        
        # Normalized if mean ~1.0 and low std
        if 0.99 <= mean_norm <= 1.01 and std_norm < 0.05:
            result["normalized"] = True
        else:
            result["normalized"] = False
        
    except Exception as e:
        result["error"] = str(e)
    
    return result


def main():
    """Validate all embedding paths."""
    
    # Define paths to check
    paths_to_check = [
        # O*NET embeddings
        {
            "type": "O*NET BGE",
            "path": "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_1/onet_embeddings/bge_large/onet_bge_large_embeddings.npy",
            "key": None
        },
        {
            "type": "O*NET E5",
            "path": "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_1/onet_embeddings/e5_large/onet_e5_large_embeddings.npy",
            "key": None
        },
        {
            "type": "O*NET GTE",
            "path": "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_1/onet_embeddings/gte_large/onet_gte_large_embeddings.npy",
            "key": None
        },
        # Target embeddings (jobs)
        {
            "type": "Target BGE",
            "path": "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2/dev/target_embeddings/bge_large/adzuna_month01/target_bge_large_month01_shard00.npz",
            "key": "embeddings"
        },
        {
            "type": "Target E5",
            "path": "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2/dev/target_embeddings/e5_large/adzuna_month01/target_e5_large_month01_shard00.npz",
            "key": "embeddings"
        },
        {
            "type": "Target GTE",
            "path": "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2/dev/target_embeddings/gte_large/adzuna_month01/target_gte_large_month01_shard00.npz",
            "key": "embeddings"
        },
    ]
    
    print("="*100)
    print("EMBEDDING PATH VALIDATION REPORT")
    print("="*100)
    print()
    
    results: List[Dict] = []
    
    for item in paths_to_check:
        print(f"Checking: {item['type']}")
        print(f"Path: {item['path']}")
        
        result = check_norms(item["path"], key=item.get("key"))
        result["type"] = item["type"]
        results.append(result)
        
        if result["exists"]:
            print(f"  ✅ EXISTS")
            print(f"  Shape: {result['shape']}")
            print(f"  Dtype: {result['dtype']}")
            if result["normalized"] is not None:
                norm_status = "✅ NORMALIZED" if result["normalized"] else "⚠️  NOT NORMALIZED"
                print(f"  {norm_status} (mean norm: {result['mean_norm']:.4f})")
        else:
            print(f"  ❌ NOT FOUND")
            if result["error"]:
                print(f"  Error: {result['error']}")
        print()
    
    # Summary table
    print("="*100)
    print("SUMMARY TABLE")
    print("="*100)
    print(f"{'Type':<20} {'Exists':<10} {'Normalized':<15} {'Shape':<20} {'Mean Norm':<12}")
    print("-"*100)
    
    for r in results:
        exists = "✅ Yes" if r["exists"] else "❌ No"
        normalized = "✅ Yes" if r["normalized"] else ("⚠️  No" if r["normalized"] is False else "N/A")
        shape = str(r["shape"]) if r["shape"] else "N/A"
        mean_norm = f"{r['mean_norm']:.4f}" if r["mean_norm"] else "N/A"
        
        print(f"{r['type']:<20} {exists:<10} {normalized:<15} {shape:<20} {mean_norm:<12}")
    
    print()
    print("="*100)
    
    # Check for issues
    issues = [r for r in results if not r["exists"] or r["error"]]
    if issues:
        print("⚠️  ISSUES FOUND:")
        for r in issues:
            print(f"  - {r['type']}: {r['error'] or 'File not found'}")
    else:
        print("✅ ALL PATHS VALID")
    
    print("="*100)


if __name__ == "__main__":
    main()